# phase_2 on Kaggle — 29-region detector + scene-graph LLM

Run with **GPU on** (Settings → Accelerator → GPU). Kaggle caps GPU sessions at
~9–12h, so run the **detector** and the **LLM** as separate sessions; both
resume from checkpoints in `/kaggle/working`.

1. Upload the `phase_2/` folder as a Kaggle dataset (or notebook utility).
2. Add your data datasets (images, scene graphs, metadata).
3. Edit the **CONFIG** cell paths, then run the section you need.

## CONFIG — edit these paths

In [ ]:
import os
# ====== EDIT to match your Kaggle dataset slugs ======
PHASE2_SRC    = "/kaggle/input/phase2-code/phase_2"     # the uploaded phase_2 folder
IMAGES        = "/kaggle/input/mimic-processed"          # MIMIC resized images (filename stem == image_id)
SCENE         = "/kaggle/input/mimic-scene-graph"        # *_SceneGraph.json files
META          = "/kaggle/input/mimic-metadata/mimic_metadata_final.jsonl"
CHEXPLUS_IMG  = "/kaggle/input/chexplus-processed"       # CheXplus images (for step 7)
CHEXPLUS_META = "/kaggle/input/chexplus-metadata/chexplpus_metadata_final.jsonl"
# =====================================================
WEIGHTS = "/kaggle/working/runs/detect/det29/weights/best.pt"
os.environ.update(dict(PHASE2_SRC=PHASE2_SRC, IMAGES=IMAGES, SCENE=SCENE, META=META,
                       CHEXPLUS_IMG=CHEXPLUS_IMG, CHEXPLUS_META=CHEXPLUS_META, WEIGHTS=WEIGHTS))
print("config set")

## SETUP — copy code into the writable dir and cd in

In [ ]:
!rm -rf /kaggle/working/phase_2 && cp -r "$PHASE2_SRC" /kaggle/working/phase_2
%cd /kaggle/working/phase_2
!ls

---
# A) DETECTOR  (run in its own GPU session)

In [ ]:
!pip install -q ultralytics

### 0 — build the YOLO dataset from scene graphs

In [ ]:
!python build_yolo_dataset.py --metadata "$META" --images-root "$IMAGES" --scene-root "$SCENE" 

### sanity — check image↔bbox alignment before training

In [ ]:
!python visualize.py --split val --n 8
from IPython.display import Image, display
import glob
for f in sorted(glob.glob("/kaggle/working/viz/*.jpg"))[:4]:
    display(Image(f))

### 1 — train YOLOv8l  (drop --imgsz/--batch if you OOM)

In [ ]:
!python train_yolo.py --imgsz 896 --batch -1 --epochs 100

**Resume** in a later session (same dataset cell + setup cell first):

In [ ]:
!python train_yolo.py --resume

### eval mAP on the held-out / gold split

In [ ]:
!python eval_yolo.py --weights "$WEIGHTS" --split test

### 2 — inference: 29-box JSON per image

In [ ]:
!python infer_yolo.py --weights "$WEIGHTS" --source "$IMAGES" --out /kaggle/working/pred --jsonl

---
# B) SCENE-GRAPH LLM  (run in its own GPU session)

In [ ]:
!pip install -q transformers trl peft bitsandbytes accelerate datasets

### 4 — controlled relation vocab

In [ ]:
!python extract_sg_vocab.py --scene-root "$SCENE" --min-count 5

### 5 — chat SFT dataset (report + regions → compact findings JSON)

In [ ]:
!python build_sft_dataset.py --metadata "$META" --scene-root "$SCENE" --keep-empty-frac 0.1

### 6 — QLoRA fine-tune Qwen2.5-7B  (single 16GB GPU; lower --max-len if OOM)

In [ ]:
!python finetune_sg_llm.py --data-dir /kaggle/working/sg_sft --out /kaggle/working/sg_lora --epochs 2

### (optional) merge LoRA → standalone weights (then use --model, drop --lora in step 7)

In [ ]:
!python merge_lora.py --lora /kaggle/working/sg_lora --out /kaggle/working/sg_merged

### 7 — pseudo scene graphs for CheXplus
First detect boxes on CheXplus images, then run the LLM. NIH is skipped (no report).

In [ ]:
!python infer_yolo.py --weights "$WEIGHTS" --source "$CHEXPLUS_IMG" --out /kaggle/working/pred_chexplus
!python build_pseudo_scene_graph.py \
  --pred-dir /kaggle/working/pred_chexplus \
  --metadata "$CHEXPLUS_META" \
  --vocab /kaggle/working/sg_vocab.json \
  --lora /kaggle/working/sg_lora \
  --out /kaggle/working/pseudo_sg \
  --update-metadata /kaggle/working/chexplus_with_scene.jsonl